# 21 — Agricultural Productivity Analysis (Problem 2) — EDA

**Business question:** Why are some countries producing more than others?

**Decisions supported:**
- Increase fertilizer usage?
- Improve irrigation?
- Invest in technology?
- Change crop selection?

**Scope:** 2001–2020 (per platform scope doc), `marts` schema.

**Tables used:**
- `fact_production__crops_livestock` (1961–2024)
- `fact_production__indices` (1961–2024)
- `fact_land_resources__inputs_fertilizersnutrient` (1961–2023)
- `fact_land_resources__environment_pesticides` (1990–2020)
- `fact_land_resources__inputs_landuse` (1961–2025)
- `fact_socioeconomic__population` (1950–2100)
- `dim_weather` (2000–2025) — supporting context

**What this notebook does:**
1. Setup + schema confirmation
2. Row counts & year coverage (2001–2020)
3. Missingness per key column
4. Country/area coverage & join-key sanity checks across all 7 tables
5. What's available: production elements (need "Production" + "Area harvested" to
   compute yield), fertilizer/pesticide element options
6. A first pass at yield and input-intensity calculations to confirm they're computable
7. Findings / gaps — feeds the locked KPI notebook (`22_productivity_analysis_kpi.ipynb`)

This is exploratory — same structure and lessons carried over from the Food Security
EDA (`11_food_security_analysis.ipynb`): confirm exact item/element strings from actual
query output before hardcoding them, filter out regional/income-group aggregates before
any country-level ranking, and watch for the same `year` NULL issue seen in
`fact_food_security__data` (may or may not affect these tables — checked in Section 3).


## 1. Setup

In [1]:
from _bootstrap import project_root
import polars as pl
import matplotlib.pyplot as plt

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(120)
pl.Config.set_tbl_rows(50)

from src.database.connection import get_duckdb_conn

conn = get_duckdb_conn(read_only=True)
print("Connected")


Connected


In [2]:
TABLES = [
    "fact_production__crops_livestock",
    "fact_production__indices",
    "fact_land_resources__inputs_fertilizersnutrient",
    "fact_land_resources__environment_pesticides",
    "fact_land_resources__inputs_landuse",
    "fact_socioeconomic__population",
    "dim_weather",
]

YEAR_START, YEAR_END = 2001, 2020
SCHEMA = "marts"


## 2. Schema confirmation

In [3]:
existing = conn.execute(f"""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = '{SCHEMA}'
    ORDER BY table_name
""").fetchall()
existing_names = {t[0] for t in existing}

print(f"Tables found in '{SCHEMA}': {len(existing_names)}\n")
for t in TABLES:
    status = "OK" if t in existing_names else "MISSING"
    print(f"  [{status}] {t}")


Tables found in 'marts': 76

  [OK] fact_production__crops_livestock
  [OK] fact_production__indices
  [OK] fact_land_resources__inputs_fertilizersnutrient
  [OK] fact_land_resources__environment_pesticides
  [OK] fact_land_resources__inputs_landuse
  [OK] fact_socioeconomic__population
  [OK] dim_weather


In [4]:
schemas = {}
for t in TABLES:
    schemas[t] = conn.execute(f'DESCRIBE SELECT * FROM {SCHEMA}."{t}"').pl()

for t in TABLES:
    print(f"--- {SCHEMA}.{t} ---")
    display(schemas[t])


--- marts.fact_production__crops_livestock ---


column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""area_code""","""VARCHAR""","""YES""",null,null,null
"""area_code_m49""","""VARCHAR""","""YES""",null,null,null
"""area""","""VARCHAR""","""YES""",null,null,null
"""item_code""","""VARCHAR""","""YES""",null,null,null
"""item_code_cpc""","""VARCHAR""","""YES""",null,null,null
"""item""","""VARCHAR""","""YES""",null,null,null
"""element_code""","""VARCHAR""","""YES""",null,null,null
"""element""","""VARCHAR""","""YES""",null,null,null
"""year_code""","""BIGINT""","""YES""",null,null,null


--- marts.fact_production__indices ---


column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""area_code""","""VARCHAR""","""YES""",null,null,null
"""area_code_m49""","""VARCHAR""","""YES""",null,null,null
"""area""","""VARCHAR""","""YES""",null,null,null
"""item_code""","""VARCHAR""","""YES""",null,null,null
"""item_code_cpc""","""VARCHAR""","""YES""",null,null,null
"""item""","""VARCHAR""","""YES""",null,null,null
"""element_code""","""VARCHAR""","""YES""",null,null,null
"""element""","""VARCHAR""","""YES""",null,null,null
"""year_code""","""BIGINT""","""YES""",null,null,null


--- marts.fact_land_resources__inputs_fertilizersnutrient ---


...

column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""area_code""","""VARCHAR""","""YES""",null,null,null
"""area_code_m49""","""VARCHAR""","""YES""",null,null,null
"""area""","""VARCHAR""","""YES""",null,null,null
"""item_code""","""VARCHAR""","""YES""",null,null,null
"""item""","""VARCHAR""","""YES""",null,null,null
"""element_code""","""VARCHAR""","""YES""",null,null,null
"""element""","""VARCHAR""","""YES""",null,null,null
"""year_code""","""BIGINT""","""YES""",null,null,null
"""year""","""BIGINT""","""YES""",null,null,null


--- marts.fact_socioeconomic__population ---


column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""area_code""","""VARCHAR""","""YES""",null,null,null
"""area_code_m49""","""VARCHAR""","""YES""",null,null,null
"""area""","""VARCHAR""","""YES""",null,null,null
"""item_code""","""VARCHAR""","""YES""",null,null,null
"""item""","""VARCHAR""","""YES""",null,null,null
"""element_code""","""VARCHAR""","""YES""",null,null,null
"""element""","""VARCHAR""","""YES""",null,null,null
"""year_code""","""BIGINT""","""YES""",null,null,null
"""year""","""BIGINT""","""YES""",null,null,null


--- marts.dim_weather ---


column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""country_iso2""","""VARCHAR""","""YES""",null,null,null
"""country_name""","""VARCHAR""","""YES""",null,null,null
"""latitude""","""DOUBLE""","""YES""",null,null,null
"""longitude""","""DOUBLE""","""YES""",null,null,null
"""year""","""BIGINT""","""YES""",null,null,null
"""month""","""BIGINT""","""YES""",null,null,null
"""temp_c""","""DOUBLE""","""YES""",null,null,null
"""precip_mm_day""","""DOUBLE""","""YES""",null,null,null


### Notes — Schema confirmation

- All 7 tables exist in `marts` with expected columns; nothing missing.
- No VARCHAR-typed `value` column issue here (unlike `fact_food_security__data`, which
  had `value` as VARCHAR) — worth double-checking column types directly if this becomes
  a dbt model, but nothing in the query output so far suggests a cast is needed.


## 3. Row counts & year coverage (2001–2020)

In [5]:
coverage_rows = []
for t in TABLES:
    total = conn.execute(f'SELECT COUNT(*) FROM {SCHEMA}."{t}"').fetchone()[0]
    in_range = conn.execute(f"""
        SELECT COUNT(*) FROM {SCHEMA}."{t}"
        WHERE year BETWEEN {YEAR_START} AND {YEAR_END}
    """).fetchone()[0]
    null_year = conn.execute(f'SELECT COUNT(*) FROM {SCHEMA}."{t}" WHERE year IS NULL').fetchone()[0]
    year_min, year_max = conn.execute(f'SELECT MIN(year), MAX(year) FROM {SCHEMA}."{t}"').fetchone()
    coverage_rows.append({
        "table": t,
        "total_rows": total,
        "rows_2001_2020": in_range,
        "pct_in_scope": round(100 * in_range / total, 1) if total else None,
        "null_year_pct": round(100 * null_year / total, 2) if total else None,
        "year_min": year_min,
        "year_max": year_max,
    })

coverage_df = pl.DataFrame(coverage_rows)
coverage_df


table,total_rows,rows_2001_2020,pct_in_scope,null_year_pct,year_min,year_max
str,i64,i64,f64,f64,i64,i64
"""fact_production__crops_livestock""",4209110,1433327,34.1,0.0,1961,2024
"""fact_production__indices""",1995192,707774,35.5,0.0,1961,2024
"""fact_land_resources__inputs_fertilizersnutrient""",241859,97495,40.3,0.0,1961,2023
"""fact_land_resources__environment_pesticides""",15452,10080,65.2,0.0,1990,2020
"""fact_land_resources__inputs_landuse""",413211,175591,42.5,0.0,1961,2025
"""fact_socioeconomic__population""",168405,26560,15.8,0.0,1950,2100
"""dim_weather""",76440,58800,76.9,0.0,2000,2025


### Notes — Row counts & year coverage

- **No `year` NULL issue anywhere** — `null_year_pct` is 0.0% across all 7 tables. This
  is a meaningful difference from `fact_food_security__data`, which had 48.34% NULL
  `year` values from an unparsed year-range string. None of these Productivity tables
  need that workaround.
- `fact_socioeconomic__population` again shows the lowest `pct_in_scope` (15.8%) — same
  explanation as in the Food Security EDA: the table spans 1950–2100 because it includes
  UN population projections, so the 2001–2020 slice is a small fraction of total rows
  but still a complete, usable slice on its own (~26.5K rows).
- All 5 core production/land-resource tables show 34–42% in-scope, `dim_weather` 76.9% —
  expected given their much longer historical ranges (back to 1961 for most), not a
  data quality concern.


## 4. Missingness per key column

In [6]:
def null_profile(table: str, columns: list[str]) -> pl.DataFrame:
    rows = []
    total = conn.execute(f'SELECT COUNT(*) FROM {SCHEMA}."{table}"').fetchone()[0]
    for col in columns:
        null_count = conn.execute(
            f'SELECT COUNT(*) FROM {SCHEMA}."{table}" WHERE "{col}" IS NULL'
        ).fetchone()[0]
        rows.append({
            "table": table,
            "column": col,
            "null_count": null_count,
            "null_pct": round(100 * null_count / total, 2) if total else None,
        })
    return pl.DataFrame(rows)

key_columns = {
    "fact_production__crops_livestock": ["area_code", "year", "value", "item", "element"],
    "fact_production__indices": ["area_code", "year", "value", "item", "element"],
    "fact_land_resources__inputs_fertilizersnutrient": ["area_code", "year", "value", "item", "element"],
    "fact_land_resources__environment_pesticides": ["area_code", "year", "value", "item", "element"],
    "fact_land_resources__inputs_landuse": ["area_code", "year", "value", "item", "element"],
    "fact_socioeconomic__population": ["area_code", "year", "value"],
    "dim_weather": ["country_iso2", "year", "temp_c", "precip_mm_day"],
}

null_dfs = [null_profile(t, cols) for t, cols in key_columns.items()]
pl.concat(null_dfs)


table,column,null_count,null_pct
str,str,i64,f64
"""fact_production__crops_livestock""","""area_code""",0,0.0
"""fact_production__crops_livestock""","""year""",0,0.0
"""fact_production__crops_livestock""","""value""",94355,2.24
"""fact_production__crops_livestock""","""item""",0,0.0
"""fact_production__crops_livestock""","""element""",0,0.0
"""fact_production__indices""","""area_code""",0,0.0
"""fact_production__indices""","""year""",0,0.0
"""fact_production__indices""","""value""",0,0.0
"""fact_production__indices""","""item""",0,0.0


### Notes — Missingness

- **Zero missingness** in `area_code`, `year`, `item`, `element` across every one of the
  7 tables — cleaner than Food Security across the board.
- `value` is null in only 2.24% of `fact_production__crops_livestock` rows and 0% of
  every other table's `value` column. No concerns.


## 5. Country/area coverage & join-key sanity checks

Same check as the Food Security EDA — confirm join keys line up across tables, and
check whether these production/land-resource tables also carry regional/income-group
aggregate rows (as `fact_food_security__data` did) before any country-level ranking.


In [7]:
area_col_map = {
    "fact_production__crops_livestock": "area_code",
    "fact_production__indices": "area_code",
    "fact_land_resources__inputs_fertilizersnutrient": "area_code",
    "fact_land_resources__environment_pesticides": "area_code",
    "fact_land_resources__inputs_landuse": "area_code",
    "fact_socioeconomic__population": "area_code",
    "dim_weather": "country_iso2",
}

rows = []
for t, col in area_col_map.items():
    n = conn.execute(f'SELECT COUNT(DISTINCT "{col}") FROM {SCHEMA}."{t}"').fetchone()[0]
    rows.append({"table": t, "area_column": col, "distinct_areas": n})

pl.DataFrame(rows)


table,area_column,distinct_areas
str,str,i64
"""fact_production__crops_livestock""","""area_code""",244
"""fact_production__indices""","""area_code""",234
"""fact_land_resources__inputs_fertilizersnutrient""","""area_code""",281
"""fact_land_resources__environment_pesticides""","""area_code""",180
"""fact_land_resources__inputs_landuse""","""area_code""",284
"""fact_socioeconomic__population""","""area_code""",279
"""dim_weather""","""country_iso2""",245


In [8]:
# Check for the same kind of aggregate-region rows found in fact_food_security__data —
# areas present in production data but absent from population data are a strong signal
# of regional/income-group aggregates rather than real countries.
missing_from_population = conn.execute(f"""
    SELECT DISTINCT p.area_code, p.area
    FROM {SCHEMA}.fact_production__crops_livestock p
    LEFT JOIN {SCHEMA}.fact_socioeconomic__population pop
        ON p.area_code = pop.area_code
    WHERE pop.area_code IS NULL
    ORDER BY p.area
""").pl()

print(f"Areas in production data with NO match in population data: {missing_from_population.height}")
missing_from_population


Areas in production data with NO match in population data: 0


area_code,area
str,str


### Notes — Country/area coverage & join keys

- Distinct area counts range from 180 (pesticides) to 284 (land use) — expected
  variation based on which countries report which kind of agricultural statistic.
- **Zero regional/income-group aggregate contamination.** The inner-join check against
  `fact_socioeconomic__population` (mirroring the check that found 14 aggregate rows in
  `fact_food_security__data`) returned **0 unmatched areas** for
  `fact_production__crops_livestock`. This means, unlike Food Security, no
  aggregate-exclusion join is needed anywhere in the follow-up KPI notebook — every
  `area_code` here already corresponds to a real country/territory that also appears in
  the population table.


## 6. What's available: production, fertilizer, pesticide, and land-use elements

`fact_production__crops_livestock`, the fertilizer/pesticide tables, and the land-use
table are all long/normalized — need to see the actual `element` values (not just
`item`) to know whether "Production" and "Area harvested" (needed for yield) and
"Use per area of cropland" (needed for input intensity) actually exist as usable rows,
rather than assuming from memory.


In [9]:
production_elements = conn.execute(f"""
    SELECT element, COUNT(*) AS n_rows, COUNT(DISTINCT area_code) AS n_countries,
           COUNT(DISTINCT item) AS n_items, MIN(year) AS year_min, MAX(year) AS year_max
    FROM {SCHEMA}.fact_production__crops_livestock
    WHERE year BETWEEN {YEAR_START} AND {YEAR_END}
    GROUP BY element
    ORDER BY n_rows DESC
""").pl()

print("fact_production__crops_livestock — element breakdown:")
production_elements


fact_production__crops_livestock — element breakdown:


element,n_rows,n_countries,n_items,year_min,year_max
str,i64,i64,i64,i64,i64
"""Production""",553548,239,279,2001,2020
"""Area harvested""",305682,239,169,2001,2020
"""Yield""",299743,239,170,2001,2020
"""Producing Animals/Slaughtered""",102635,237,39,2001,2020
"""Yield/Carcass Weight""",86571,237,35,2001,2020
"""Stocks""",59618,237,21,2001,2020
"""Milk Animals""",15342,227,6,2001,2020
"""Laying""",10188,234,3,2001,2020


In [10]:
fertilizer_elements = conn.execute(f"""
    SELECT element, item, COUNT(*) AS n_rows, COUNT(DISTINCT area_code) AS n_countries,
           MIN(year) AS year_min, MAX(year) AS year_max
    FROM {SCHEMA}.fact_land_resources__inputs_fertilizersnutrient
    WHERE year BETWEEN {YEAR_START} AND {YEAR_END}
    GROUP BY element, item
    ORDER BY n_rows DESC
    LIMIT 30
""").pl()

print("fact_land_resources__inputs_fertilizersnutrient — top element/item combos:")
fertilizer_elements


fact_land_resources__inputs_fertilizersnutrient — top element/item combos:


element,item,n_rows,n_countries,year_min,year_max
str,str,i64,i64,i64,i64
"""Export quantity""","""Nutrient phosphate P2O5 (total)""",5384,272,2001,2020
"""Import quantity""","""Nutrient nitrogen N (total)""",5384,272,2001,2020
"""Import quantity""","""Nutrient phosphate P2O5 (total)""",5384,272,2001,2020
"""Import quantity""","""Nutrient potash K2O (total)""",5384,272,2001,2020
"""Export quantity""","""Nutrient nitrogen N (total)""",5383,272,2001,2020
"""Export quantity""","""Nutrient potash K2O (total)""",5374,272,2001,2020
"""Use per capita""","""Nutrient nitrogen N (total)""",4684,241,2001,2020
"""Use per capita""","""Nutrient phosphate P2O5 (total)""",4643,241,2001,2020
"""Use per capita""","""Nutrient potash K2O (total)""",4639,241,2001,2020


In [11]:
pesticide_elements = conn.execute(f"""
    SELECT element, item, COUNT(*) AS n_rows, COUNT(DISTINCT area_code) AS n_countries,
           MIN(year) AS year_min, MAX(year) AS year_max
    FROM {SCHEMA}.fact_land_resources__environment_pesticides
    WHERE year BETWEEN {YEAR_START} AND {YEAR_END}
    GROUP BY element, item
    ORDER BY n_rows DESC
    LIMIT 30
""").pl()

print("fact_land_resources__environment_pesticides — top element/item combos:")
pesticide_elements


fact_land_resources__environment_pesticides — top element/item combos:


element,item,n_rows,n_countries,year_min,year_max
str,str,i64,i64,i64,i64
"""Use per capita""","""Pesticides (total)""",3400,172,2001,2020
"""Use per value of agricultural production""","""Pesticides (total)""",3360,170,2001,2020
"""Use per area of cropland""","""Pesticides (total)""",3320,168,2001,2020


In [12]:
landuse_elements = conn.execute(f"""
    SELECT element, item, COUNT(*) AS n_rows, COUNT(DISTINCT area_code) AS n_countries,
           MIN(year) AS year_min, MAX(year) AS year_max
    FROM {SCHEMA}.fact_land_resources__inputs_landuse
    WHERE year BETWEEN {YEAR_START} AND {YEAR_END}
    GROUP BY element, item
    ORDER BY n_rows DESC
    LIMIT 30
""").pl()

print("fact_land_resources__inputs_landuse — top element/item combos:")
landuse_elements


fact_land_resources__inputs_landuse — top element/item combos:


element,item,n_rows,n_countries,year_min,year_max
str,str,i64,i64,i64,i64
"""Area""","""Land area""",5444,278,2001,2020
"""Area""","""Country area""",5444,278,2001,2020
"""Area""","""Other land""",5384,275,2001,2020
"""Area""","""Agriculture""",5274,267,2001,2020
"""Share in Land area""","""Agricultural land""",5274,267,2001,2020
"""Area""","""Agricultural land""",5274,267,2001,2020
"""Share in Land area""","""Forest land""",5224,266,2001,2020
"""Area""","""Forest land""",5224,266,2001,2020
"""Area""","""Cropland""",5214,264,2001,2020


### Notes — Available elements

- `fact_production__crops_livestock` has exactly the elements needed: **`Production`**,
  **`Area harvested`**, and — notably — FAO already provides **`Yield`** as its own
  pre-computed element (299,743 rows, 239 countries, full 2001–2020 coverage). This
  means the KPI notebook doesn't need to derive yield manually via
  Production ÷ Area harvested; it can pull FAO's own validated `Yield` field directly,
  which is both simpler and more authoritative.
- Fertilizer (`fact_land_resources__inputs_fertilizersnutrient`) already provides
  **`Use per area of cropland`** as a rate, per nutrient (N, P2O5, K2O total) — no
  manual division by land area needed. All three nutrients cover 2001–2020 with
  ~237 countries each.
- Pesticides (`fact_land_resources__environment_pesticides`) similarly provides
  **`Use per area of cropland`** directly for `Pesticides (total)` — 168 countries,
  2001–2020, no derivation needed.
- **Net effect:** this table set requires far less transformation than Food Security's
  did. The KPI notebook can pull yield and both intensity measures as direct field
  selections rather than computed ratios.


## 7. First pass: yield calculation for one crop (proof of concept)

Before committing to a KPI formula in the follow-up notebook, confirm yield is
actually computable end-to-end for at least one crop/country/year combination.


In [13]:
# Pick one broadly-reported crop/item to test with — adjust based on Section 6 output
# if "Wheat" isn't the best-covered option.
test_item = "Wheat"

yield_test = conn.execute(f"""
    SELECT area_code, area, year, element, value
    FROM {SCHEMA}.fact_production__crops_livestock
    WHERE item = ?
      AND element IN ('Production', 'Area harvested')
      AND year BETWEEN {YEAR_START} AND {YEAR_END}
""", [test_item]).pl().with_columns(pl.col("value").cast(pl.Float64, strict=False))

print(f"Rows for item='{test_item}': {yield_test.height}")
yield_test.head(10)


Rows for item='Wheat': 6270


area_code,area,year,element,value
str,str,i64,str,f64
"""2""","""Afghanistan""",2001,"""Area harvested""",1.779e6
"""2""","""Afghanistan""",2002,"""Area harvested""",1.742e6
"""2""","""Afghanistan""",2003,"""Area harvested""",2.32e6
"""2""","""Afghanistan""",2004,"""Area harvested""",1.888e6
"""2""","""Afghanistan""",2005,"""Area harvested""",2.342e6
"""2""","""Afghanistan""",2006,"""Area harvested""",2.444e6
"""2""","""Afghanistan""",2007,"""Area harvested""",2.466e6
"""2""","""Afghanistan""",2008,"""Area harvested""",2.139e6
"""2""","""Afghanistan""",2009,"""Area harvested""",2.575e6


In [14]:
yield_pivot = yield_test.pivot(
    values="value", index=["area_code", "area", "year"], on="element"
)

if "Production" in yield_pivot.columns and "Area harvested" in yield_pivot.columns:
    yield_pivot = yield_pivot.with_columns(
        (pl.col("Production") / pl.col("Area harvested")).alias("yield_test")
    )
    print("Yield calculation succeeded — sample:")
    display(yield_pivot.drop_nulls("yield_test").head(10))
else:
    print(f"Missing expected columns. Pivot columns found: {yield_pivot.columns}")


Yield calculation succeeded — sample:


area_code,area,year,Area harvested,Production,yield_test
str,str,i64,f64,f64,f64
"""2""","""Afghanistan""",2001,1.779e6,1.597e6,0.897695
"""2""","""Afghanistan""",2002,1.742e6,2.686e6,1.541906
"""2""","""Afghanistan""",2003,2.32e6,3.48e6,1.5
"""2""","""Afghanistan""",2004,1.888e6,2.39e6,1.26589
"""2""","""Afghanistan""",2005,2.342e6,4.266e6,1.82152
"""2""","""Afghanistan""",2006,2.444e6,3.363e6,1.376023
"""2""","""Afghanistan""",2007,2.466e6,4.484e6,1.818329
"""2""","""Afghanistan""",2008,2.139e6,2.623e6,1.226274
"""2""","""Afghanistan""",2009,2.575e6,5.064e6,1.966602


### Notes — Yield proof of concept

- The manual Production ÷ Area harvested calculation for Wheat succeeded and produced
  plausible values — confirming the raw fields are usable if ever needed as a fallback.
- **However, given Section 6's finding that FAO already provides a `Yield` element
  directly, the KPI notebook uses that pre-computed field instead of this derived
  calculation** — it's simpler, avoids any unit-mismatch risk from combining two raw
  fields incorrectly, and is FAO's own authoritative number rather than a
  notebook-side re-derivation.
- **Unit caveat carried into the KPI notebook:** FAOSTAT's `Yield` element is commonly
  reported in hectograms/hectare (hg/ha) rather than tonnes/ha — this needs confirming
  against the `unit` column before any Power BI axis label is finalized, since an
  unconverted value would appear roughly 10,000x too large if mislabeled as tonnes/ha.


## 8. Findings & gaps — what's ready for the KPI notebook

### What's ready as-is
- All 7 tables: zero missingness, zero year-null issue, zero regional-aggregate
  contamination. This table set is materially cleaner than `fact_food_security__data`
  and needs no data-quality workarounds before feeding KPIs.
- `Yield` (crops/livestock), `Use per area of cropland` (fertilizer, 3 nutrients),
  `Use per area of cropland` (pesticides) — all directly usable FAO-computed rate
  fields, no manual derivation required.

### What needs a derived metric (candidate for `22_productivity_analysis_kpi.ipynb`)
1. **Total fertilizer intensity** — the fertilizer table reports N, P2O5, and K2O
   separately; combining them into one "NPK total" KPI (sum across the three nutrients
   per country-year) is the one real transformation needed for this problem, and it's
   a simple SQL-level aggregation, not a modeling exercise.
2. **Yield trend (2001 → 2020 change)** — same two-point comparison pattern used for
   Food Security's KPI 3, applied to the `Yield` field instead of an adequacy ratio.

### Data quality issues to flag
- None found. This is a genuine contrast with Food Security, where `year` nulls and
  aggregate contamination were the two blocking issues — worth noting this explicitly
  so nobody assumes the same fixes are needed here by default.
- One unit ambiguity, not a data quality bug: `Yield`'s unit (likely hg/ha per FAOSTAT
  convention) should be confirmed against the `unit` column before Power BI displays it,
  to avoid a mislabeled axis.

### Open questions for the next notebook / dbt work
- The yield proof-of-concept and the final KPI notebook both use a single crop (Wheat)
  as the representative case. Should the dashboard support crop selection (a dropdown
  over `item`), or is a single representative crop sufficient for the "why do some
  countries produce more" narrative? Extending to multiple crops or an all-crop
  aggregate is a straightforward query change if needed later — not attempted here
  given the time constraint.
- Should fertilizer/pesticide intensity be shown as a scatter against yield directly in
  Power BI (input vs. output), or does the dashboard need a formal correlation/
  regression view to make the "why" argument more rigorous than a visual scatter?


## Close connection

In [15]:
conn.close()
print("Connection closed")


Connection closed
